Q2.
(i) ITT estimand means the difference in the outcomes between the treated and the control group at an average level. It captures the causal effect of being assigned to treatment regardless of whether the treatment was sucessful.
(ii) ITT is the default estimand because it reflects the real-world impact of deploying the treatment at scale, such as fail to deliver, non-engagement, attributions. ITT provides the most policy-relevant estimate of the overall impact of the intervention because it affects all assigned users.
(iii) However, ITT may not be the estimand of interest if we want to measure the causal effect of those who actually receiving the treatment, rather than simply being assigned to it. For example, if some users assigned to treatment didn't engage with it, ITT will understate the true effect of treatment among those who actually experience it. In this case, researchers may target the Treatment-on-the-Treated (TOT) or Local Average Treatment Effect (LATE), which measures the effect among compliers.

In [1]:
############### simulate users table ##########
# load necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy.special as sc
import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(123)

###############################################
# set up directory
###############################################

# Create necessary directories for our project
os.makedirs("/Users/a75700/Desktop/soda_501/soda_501/experiment/data/raw", exist_ok=True)         # For raw, unprocessed data (e.g., event logs)
os.makedirs("/Users/a75700/Desktop/soda_501/soda_501/experiment/data/processed", exist_ok=True)   # For cleaned, processed data (analysis-ready)
os.makedirs("/Users/a75700/Desktop/soda_501/soda_501/experiment/outputs/figures", exist_ok=True)  # For plots and visualizations
os.makedirs("/Users/a75700/Desktop/soda_501/soda_501/experiment/outputs/tables", exist_ok=True)   # For analysis results and summaries

###############################################
# LOGGING SETUP
###############################################

# Minimal logging (file + console) without functions
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("analysis_log.txt", mode="w"),
        logging.StreamHandler()
    ]
)

###############################################
# EXPERIMENT PIPELINE (SEQUENTIAL)
###############################################

###############################################
# STEP 0: GLOBAL SETTINGS
###############################################

# Reproducibility seed
np.random.seed(123)

# "Big data" knobs
n_users = 100000     # number of users in the experiment
n_days  = 14         # number of post-assignment days to log

logging.info("Starting big data experiment pipeline")
logging.info(f"n_users = {n_users} | n_days = {n_days}")

###############################################
# STEP 1: GENERATE SYNTHETIC USERS (UNIT TABLE)
###############################################

logging.info("Generating synthetic user table")

user_id = np.arange(1, n_users + 1)
# create a variable of the platform
platform = np.random.choice(
    ["ios", "android", "web"],
    size=n_users,
    replace=True,
    p=[0.35, 0.35, 0.30]
) # with 35% using ios, 35% using android, and 30% web

cluster_id = np.random.randint(1, 501, size=n_users)

baseline_activity = np.random.gamma(shape=2.0, scale=2.0, size=n_users)

signup_cohort = np.random.choice(
    ["cohort_A", "cohort_B", "cohort_C"],
    size=n_users,
    replace=True,
    p=[0.40, 0.35, 0.25]
)

users = pd.DataFrame({
    "user_id": user_id,
    "platform": platform,
    "cluster_id": cluster_id,
    "baseline_activity": baseline_activity,
    "signup_cohort": signup_cohort
})

# Pre-treatment metric (placebo outcome) correlated with baseline_activity
users["pre_metric"] = users["baseline_activity"] + np.random.normal(0, 0.5, size=n_users)

# Save raw user table
users.to_csv("/Users/a75700/Desktop/soda_501/soda_501/experiment/data/raw/users.csv", index=False)
logging.info("/Users/a75700/Desktop/soda_501/soda_501/experiment/data/raw/users.csv")


2026-02-15 21:10:41,050 | INFO | Starting big data experiment pipeline
2026-02-15 21:10:41,051 | INFO | n_users = 100000 | n_days = 14
2026-02-15 21:10:41,051 | INFO | Generating synthetic user table
2026-02-15 21:10:41,215 | INFO | /Users/a75700/Desktop/soda_501/soda_501/experiment/data/raw/users.csv


In [3]:
##### Create assignment df #########
logging.info("Creating blocked assignment table (and saving it)")

# Blocking: deciles of baseline activity
users["block"] = pd.qcut(users["baseline_activity"], 10, labels=False) + 1  # 1..10

# Create variables named treated
# (groupby + transform returns aligned vector; no functions defined)
users["treat"] = (
    users.groupby("block")["user_id"]
    .transform(lambda s: (np.random.rand(len(s)) < 0.7).astype(int))
)
##### the probability of being treated is 0.7n
assignment = users[[
    "user_id", "treat", "block", "platform", "cluster_id",
    "signup_cohort", "baseline_activity", "pre_metric"
]].copy()

assignment["assignment_date"] = np.datetime64("2026-04-16")

# SAVE assignment table (essential reproducibility artifact)
assignment.to_csv("/Users/a75700/Desktop/soda_501/soda_501/experiment/data/raw/assignment_table.csv", index=False)
logging.info("Saved: data/raw/assignment_table.csv")


2026-02-15 22:39:41,439 | INFO | Creating blocked assignment table (and saving it)
2026-02-15 22:39:41,749 | INFO | Saved: data/raw/assignment_table.csv


In [13]:
##### Q4. retention-style outcome and estimate ATE ########
n_days = 14

# expand to user-day
logs = (
    assignment[["user_id", "treat", "baseline_activity", "cluster_id", "block"]]
    .assign(key=1)
    .merge(pd.DataFrame({"day": np.arange(1, n_days + 1), "key": 1}), on="key")
    .drop(columns="key")
)

# probability of being active each day depends on baseline + treatment
x = (logs["baseline_activity"] - logs["baseline_activity"].mean()) / logs["baseline_activity"].std()
p = sc.expit(-1.2 + 0.6 * x + 0.25 * logs["treat"])

logs["active"] = (np.random.rand(len(logs)) < p).astype(int)

# create active day, retention outcomes from user-day logs
retention = (
    logs.groupby("user_id")["active"]
    .sum()
    .reset_index(name="active_days")
)

retention["retained_any"] = (retention["active_days"] >= 1).astype(int)

# merge altogether
analysis = assignment.merge(retention, on="user_id", how="left")
analysis["active_days"] = analysis["active_days"].fillna(0)
analysis["retained_any"] = analysis["retained_any"].fillna(0)

analysis.to_csv(
    "/Users/a75700/Desktop/soda_501/soda_501/experiment/data/processed/analysis_ready.csv",
    index=False
)
logging.info("Saved analysis_ready.csv")

# diff in means
ate_diff = (
    analysis.groupby("treat")["retained_any"]
    .mean()
)

ate_diff_means = ate_diff[1] - ate_diff[0]

print("ATE (difference in means):", ate_diff_means)

# regression adjustment with factor
model = smf.ols(
    "retained_any ~ treat + C(block)",
    data=analysis
).fit(
    cov_type="cluster",
    cov_kwds={"groups": analysis["cluster_id"]}
)
#ate
ate_reg = model.params["treat"]
# cluster robust SE
se_cluster = model.bse["treat"]
print("ATE (regression):", ate_reg)
print("Cluster-robust SE:", se_cluster)

# save results
results = pd.DataFrame({
    "ate_diff_means": [ate_diff_means],
    "ate_regression": [ate_reg],
    "se_clustered": [se_cluster]
})

results.to_csv(
    "/Users/a75700/Desktop/soda_501/soda_501/experiment/outputs/tables/ate_retention.csv",
    index=False
)

logging.info("Saved ate_retention.csv")


2026-02-17 09:42:32,778 | INFO | Saved analysis_ready.csv
2026-02-17 09:42:32,926 | INFO | Saved ate_retention.csv


ATE (difference in means): 0.021547619047619038
ATE (regression): 0.02181318748094355
Cluster-robust SE: 0.0013106900930326898


In [14]:
# create a variable named 'received', controls have received 0,
# treated have received = 1 with prob = 0.8
p_receive = 0.8

users["received"] = (
    users["treat"] &
    (np.random.rand(len(users)) < p_receive)
).astype(int)

# redefine DV so that the treatment effect operates
# through received
beta = 5 # define treatment effect
users["outcome"] = (
    users["baseline_activity"]
    + beta * users["received"]
    + np.random.normal(0, 1, len(users))
)

# ITT
X = sm.add_constant(users["treat"])
itt = sm.OLS(users["outcome"], X).fit()
itt.params["treat"]

#
itt = users.loc[users.treat==1, "outcome"].mean() - users.loc[users.treat==0, "outcome"].mean()
fs  = users.loc[users.treat==1, "received"].mean() - users.loc[users.treat==0, "received"].mean()

late = itt / fs
itt, fs, late

# save

res = pd.DataFrame([{
    "ITT": float(itt),
    "first_stage": float(fs),
    "TOT_IV_LATE": float(late)
}])

res.to_csv("/Users/a75700/Desktop/soda_501/soda_501/experiment/outputs/tables/itt_vs_tot.csv", index=False)


I set the treatment effect to β = 5, which represents the causal effect of receiving treatment on the outcome. However, not all individuals assigned to treatment actually received it, as shown by the first stage estimate of 0.798. As a result, the ITT estimate (3.959) reflects the average effect of assignment, which is diluted by noncompliance. The TOT estimate rescales the ITT by the compliance rate and recovers the causal effect among compliers, resulting in an estimate (4.962) close to the true treatment effect of 5. Therefore, TOT is larger than ITT because ITT averages over both treated and untreated individuals within the assigned group.